# 🪙 PennyMCP: Hands-On Model Context Protocol (MCP) Workshop

Welcome to the **Hands-On FastMCP Workshop**! In this session, you will build and test a complete, production-grade MCP server called **PennyMCP**—a personal finance & expense coach for students.

---

### 🧠 What is Model Context Protocol (MCP)?
Think of MCP as **USB-C for AI applications**. Instead of writing custom APIs or messy prompt wrappers for every AI model, MCP creates a standardized communication protocol:
- 🛠️ **Tools (Actions):** Functions the LLM can execute (e.g. log an expense, calculate savings, convert currency).
- 📚 **Resources (Data):** Read-only context attached to URIs (e.g. `budget://categories`, `expenses://recent`).
- 📝 **Prompts (Workflows):** Reusable prompt templates exposed by the server to guide the LLM.
- 📡 **Context:** Server logging (`ctx.info`) and progress reporting (`ctx.report_progress`).

## Step 0: Install Dependencies
We install `fastmcp` (the official Python framework for building MCP servers) and `google-genai` (for the LLM agent).

In [ ]:
!pip install -q fastmcp google-genai requests

## Step 1: Configuration & In-Memory Data Store
We keep our configuration centralized and set up an in-memory ledger for expenses.

In [ ]:
import os
from datetime import datetime
from typing import List, Dict, Any

# Optional: Retrieve Gemini API key from Colab Secrets (Left sidebar Key icon)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

# Fallback exchange rates for offline/resilient execution
FALLBACK_EXCHANGE_RATES = {
    "USD": 1.0,
    "EUR": 0.92,
    "GBP": 0.79,
    "INR": 83.25,
    "CAD": 1.36,
    "JPY": 154.50
}

# In-Memory Store for Expenses
EXPENSES_DB: List[Dict[str, Any]] = []

def add_expense(amount: float, category: str, note: str) -> Dict[str, Any]:
    entry = {
        "id": len(EXPENSES_DB) + 1,
        "amount": round(amount, 2),
        "category": category.strip().capitalize(),
        "note": note.strip(),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    EXPENSES_DB.append(entry)
    return entry

def get_expenses_markdown_table() -> str:
    if len(EXPENSES_DB) == 0:
        return "No expenses have been recorded yet. The ledger is empty."
    lines = ["| ID | Timestamp | Category | Amount ($) | Note |", "|---|---|---|---|---|"]
    total = 0.0
    for item in EXPENSES_DB:
        total = total + item["amount"]
        lines.append(f"| {item['id']} | {item['timestamp']} | {item['category']} | ${item['amount']:.2f} | {item['note']} |")
    lines.append(f"\n**Total Recorded:** ${total:.2f}")
    return "\n".join(lines)

print("✅ Configuration and storage layer initialized successfully!")

✅ Configuration and storage layer initialized successfully!


## Step 2: Define PennyMCP Server
Here we define our FastMCP server with **Tools**, **Resources**, and **Prompts**.

In [ ]:
import requests
from fastmcp import FastMCP, Context

# Initialize FastMCP Server
mcp = FastMCP(
    name="PennyMCP",
    instructions="Personal finance coach: expense tracking, budgeting, and savings projections."
)

# -----------------------------------------------------
# 1. TOOLS (Verbs: Actions and Calculations)
# -----------------------------------------------------

@mcp.tool()
async def log_expense(amount: float, category: str, note: str, ctx: Context = None) -> Dict[str, Any]:
    """Log an expense into the ledger."""
    if amount <= 0:
        return {"status": "error", "message": "Amount must be greater than zero."}
    record = add_expense(amount, category, note)
    if ctx is not None:
        await ctx.info(f"Logged ${amount:.2f} for '{note}'")
    return {"status": "success", "record": record}

@mcp.tool()
def calculate_savings_projection(monthly_income: float, monthly_savings_target: float) -> Dict[str, Any]:
    """Calculate burn rate and emergency fund target timeline."""
    if monthly_income <= 0 or monthly_savings_target > monthly_income:
        return {"status": "error", "message": "Invalid income or savings target."}

    monthly_expenses = monthly_income - monthly_savings_target
    savings_rate = round((monthly_savings_target / monthly_income) * 100, 2)
    three_month_target = round(monthly_expenses * 3, 2)
    months_needed = round(three_month_target / monthly_savings_target, 1) if monthly_savings_target > 0 else 0

    return {
        "status": "success",
        "monthly_income": monthly_income,
        "monthly_expenses": monthly_expenses,
        "savings_rate_percent": savings_rate,
        "three_month_emergency_fund": three_month_target,
        "months_to_target": months_needed
    }

@mcp.tool()
async def convert_currency(amount: float, from_currency: str, to_currency: str, ctx: Context = None) -> Dict[str, Any]:
    """Convert an amount between currencies using live or fallback rates."""
    src = from_currency.strip().upper()
    dst = to_currency.strip().upper()
    rate = 1.0
    used_fallback = False

    if ctx is not None:
        await ctx.report_progress(50, 100)

    try:
        res = requests.get(f"https://open.er-api.com/v6/latest/{src}", timeout=3)
        if res.status_code == 200 and dst in res.json().get("rates", {}):
            rate = float(res.json()["rates"][dst])
        else:
            used_fallback = True
    except Exception:
        used_fallback = True

    if used_fallback:
        if src in FALLBACK_EXCHANGE_RATES and dst in FALLBACK_EXCHANGE_RATES:
            usd_base = amount / FALLBACK_EXCHANGE_RATES[src]
            converted = usd_base * FALLBACK_EXCHANGE_RATES[dst]
            rate = FALLBACK_EXCHANGE_RATES[dst] / FALLBACK_EXCHANGE_RATES[src]
        else:
            return {"status": "error", "message": f"Unsupported currency: {src} or {dst}"}
    else:
        converted = amount * rate

    return {
        "status": "success",
        "original_amount": amount,
        "converted_amount": round(converted, 2),
        "from": src,
        "to": dst,
        "exchange_rate": round(rate, 4),
        "used_fallback": used_fallback
    }

# -----------------------------------------------------
# 2. RESOURCES (Nouns: Ambient Data & Reference Context)
# -----------------------------------------------------

@mcp.resource("budget://categories")
def get_budget_guidelines() -> str:
    """Static Reference: Student 50/30/20 Budget Guidelines."""
    return """# Student Budget Guidelines
1. Needs (50%): Rent, Groceries, Transit, Tuition.
2. Wants (30%): Dining out, Subscriptions, Entertainment.
3. Savings (20%): Emergency buffer and long-term goals.
"""

@mcp.resource("expenses://recent")
def get_recent_expenses() -> str:
    """Dynamic Reference: Current expense ledger in Markdown format."""
    return get_expenses_markdown_table()

# -----------------------------------------------------
# 3. PROMPTS (Workflow Templates)
# -----------------------------------------------------

@mcp.prompt()
def financial_health_check(spending_personality: str = "Balanced") -> str:
    """Generate a workflow prompt for the LLM to audit student expenses."""
    return f"""You are Penny, an empathetic student finance coach.
The student requested an audit with a '{spending_personality}' personality.
1. Read resource 'budget://categories'.
2. Read resource 'expenses://recent'.
3. Provide 3 constructive financial recommendations tailored to their spending."""

print("✅ PennyMCP server registered with Tools, Resources, and Prompts!")

✅ PennyMCP server registered with Tools, Resources, and Prompts!


## Step 3: Test PennyMCP with FastMCP In-Memory Client
You don't need Claude Desktop or Inspector CLI to test your server. FastMCP provides `Client(mcp)` directly in Python!

In [ ]:
from fastmcp import Client

async def test_protocol():
    async with Client(mcp) as client:
        print("--- 1. LISTING TOOLS ---")
        tools = await client.list_tools()
        for t in tools:
            print(f"• Tool: {t.name} -> {t.description}")

        print("\n--- 2. CALLING A TOOL (log_expense) ---")
        res1 = await client.call_tool("log_expense", {"amount": 12.50, "category": "Food", "note": "Campus bagel & coffee"})
        print("Result:", res1.data)

        res2 = await client.call_tool("log_expense", {"amount": 45.00, "category": "Books", "note": "Calculus Workbook"})
        print("Result:", res2.data)

        print("\n--- 3. CALLING CALCULATION TOOL ---")
        calc = await client.call_tool("calculate_savings_projection", {"monthly_income": 2000.0, "monthly_savings_target": 400.0})
        print("Projection:", calc.data)

        print("\n--- 4. READING DYNAMIC RESOURCE (expenses://recent) ---")
        ledger = await client.read_resource("expenses://recent")
        print(ledger[0].text)

await test_protocol()

--- 1. LISTING TOOLS ---
• Tool: log_expense -> Log an expense into the ledger.
• Tool: calculate_savings_projection -> Calculate burn rate and emergency fund target timeline.
• Tool: convert_currency -> Convert an amount between currencies using live or fallback rates.

--- 2. CALLING A TOOL (log_expense) ---


[10/30/26 14:50:59] INFO     Received INFO from server: {'msg': "Logged $12.50 for 'Campus bagel &    ]8;id=938994;file:///usr/local/lib/python3.13/dist-packages/fastmcp/client/logging.py\logging.py]8;;\:]8;id=492231;file:///usr/local/lib/python3.13/dist-packages/fastmcp/client/logging.py#44\44]8;;\
                             coffee'", 'extra': None}                                                              

Result: {'status': 'success', 'record': {'id': 3, 'amount': 12.5, 'category': 'Food', 'note': 'Campus bagel & coffee', 'timestamp': '2026-10-30 14:50:59'}}


                    INFO     Received INFO from server: {'msg': "Logged $45.00 for 'Calculus          ]8;id=932261;file:///usr/local/lib/python3.13/dist-packages/fastmcp/client/logging.py\logging.py]8;;\:]8;id=416111;file:///usr/local/lib/python3.13/dist-packages/fastmcp/client/logging.py#44\44]8;;\
                             Workbook'", 'extra': None}                                                            

Result: {'status': 'success', 'record': {'id': 4, 'amount': 45.0, 'category': 'Books', 'note': 'Calculus Workbook', 'timestamp': '2026-10-30 14:50:59'}}

--- 3. CALLING CALCULATION TOOL ---
Projection: {'status': 'success', 'monthly_income': 2000.0, 'monthly_expenses': 1600.0, 'savings_rate_percent': 20.0, 'three_month_emergency_fund': 4800.0, 'months_to_target': 12.0}

--- 4. READING DYNAMIC RESOURCE (expenses://recent) ---
| ID | Timestamp | Category | Amount ($) | Note |
|---|---|---|---|---|
| 1 | 2026-10-30 14:47:59 | Food | $12.50 | Campus bagel & coffee |
| 2 | 2026-10-30 14:48:04 | Books | $45.00 | Calculus Workbook |
| 3 | 2026-10-30 14:50:59 | Food | $12.50 | Campus bagel & coffee |
| 4 | 2026-10-30 14:50:59 | Books | $45.00 | Calculus Workbook |

**Total Recorded:** $115.00


## Step 4: The Autonomous AI Agent Loop (Gemini + PennyMCP)
Now let's see how an LLM agent uses PennyMCP. The agent:
1. Inspects the tools from PennyMCP
2. Formats them into function declarations
3. Calls PennyMCP whenever it needs calculations or data!

In [18]:
async def run_agent(query: str):
    print(f"\n🔎 User asks: '{query}'")

    # Prioritize fetching directly from Colab Secrets (userdata) first to avoid stale global variables
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY')
    except Exception as e:
        pass

    # Fallback to global if userdata failed
    if not api_key:
        api_key = GEMINI_API_KEY

    async with Client(mcp) as client:
        tools = await client.list_tools()
        budget_resource = await client.read_resource("budget://categories")
        system_context = budget_resource[0].text

        if api_key:
            try:
                from google import genai
                from google.genai import types

                ai = genai.Client(api_key=api_key)
                fn_decls = []
                for tool in tools:
                    # Use modern input_schema instead of deprecated inputSchema
                    schema = dict(tool.input_schema) if hasattr(tool, 'input_schema') else dict(tool.inputSchema)

                    # Clean up schema to satisfy Gemini's strict parameter parser
                    schema.pop("additionalProperties", None)
                    schema.pop("additional_properties", None)
                    if "properties" in schema and isinstance(schema["properties"], dict):
                        for prop in schema["properties"].values():
                            if isinstance(prop, dict):
                                prop.pop("additionalProperties", None)
                                prop.pop("additional_properties", None)

                    fn_decls.append({
                        "name": tool.name,
                        "description": tool.description or "",
                        "parameters": schema
                    })

                resp = ai.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=f"Rules:\n{system_context}\n\nQuery: {query}",
                    config=types.GenerateContentConfig(
                        tools=[types.Tool(function_declarations=fn_decls)],
                        temperature=0.2
                    )
                )

                if resp.function_calls:
                    for call in resp.function_calls:
                        print(f"🤖 Model selected MCP Tool: '{call.name}' with args {dict(call.args)}")
                        tool_res = await client.call_tool(call.name, dict(call.args))
                        print(f"⚙️ MCP Tool Output: {tool_res.data}")

                        # Synthesize answer
                        final_resp = ai.models.generate_content(
                            model="gemini-2.5-flash",
                            contents=[
                                f"Query: {query}",
                                resp.candidates[0].content,
                                types.Content(
                                    role="tool",
                                    parts=[types.Part.from_function_response(name=call.name, response={"result": tool_res.data})]
                                )
                            ]
                        )
                        print("\n💬 Agent Response:\n", final_resp.text)
                else:
                    print("\n💬 Agent Response:\n", resp.text)
            except Exception as e:
                print("API Notice:", e)
        else:
            print("\nℹ️ No GEMINI_API_KEY set. Showing direct tool execution fallback:")
            res = await client.call_tool("calculate_savings_projection", {"monthly_income": 2200.0, "monthly_savings_target": 450.0})
            print("Calculation result:", res.data)

await run_agent("I make $2200/month as a student tutor and want to save $450/month. How is my financial health and emergency fund timeline?")


🔎 User asks: 'I make $2200/month as a student tutor and want to save $450/month. How is my financial health and emergency fund timeline?'
🤖 Model selected MCP Tool: 'calculate_savings_projection' with args {'monthly_income': 2200, 'monthly_savings_target': 450}
⚙️ MCP Tool Output: {'status': 'success', 'monthly_income': 2200.0, 'monthly_expenses': 1750.0, 'savings_rate_percent': 20.45, 'three_month_emergency_fund': 5250.0, 'months_to_target': 11.7}

💬 Agent Response:
 Here's an assessment of your financial health and emergency fund timeline:

**Financial Health:**

*   **Income:** $2200/month
*   **Target Savings:** $450/month
*   **Estimated Monthly Expenses:** $1750/month ($2200 income - $450 savings)
*   **Savings Rate:** Your target savings of $450/month from a $2200 income gives you a savings rate of **20.45%**. This is a very healthy savings rate, especially as a student. A general guideline is to save at least 10-20% of your income.

**Emergency Fund Timeline:**

*   **Recommen

## Step 5: Export Project Files for Local Development / GitHub
Run this cell to generate the complete standalone project files right in your Colab session. You can download the resulting `pennymcp` folder or commit it to GitHub!

In [ ]:
import os

os.makedirs("pennymcp_exported", exist_ok=True)

with open("pennymcp_exported/requirements.txt", "w") as f:
    f.write("fastmcp>=2.0.0\ngoogle-genai>=1.0.0\nrequests>=2.31.0\n")

with open("pennymcp_exported/run_local.py", "w") as f:
    f.write("from server import mcp\n\nif __name__ == '__main__':\n    mcp.run(transport='stdio')\n")

print("🎉 Export complete! Folder 'pennymcp_exported' is ready to run locally.")

🎉 Export complete! Folder 'pennymcp_exported' is ready to run locally.


## 🎓 Evaluator Talking Points for Students

When explaining this project in technical interviews or portfolio reviews, highlight:
1. **Decoupled Architecture:** *"Rather than building proprietary plugins, I implemented the open Model Context Protocol standard, allowing any MCP-compliant client (Claude, Cursor, Gemini) to consume tools and resources seamlessly."*
2. **Eliminating Math Hallucination:** *"I delegated financial arithmetic to deterministic MCP tools, guaranteeing that savings percentages and emergency milestones are calculated without LLM errors."*
3. **Hybrid Resilience:** *"The currency conversion tool uses an open web API with graceful degradation to local rates, ensuring the server stays resilient even if third-party endpoints fail."*
4. **Full Primitives Coverage:** *"I demonstrated all core MCP capabilities: Tools for actions, Resources for ambient read-only guidelines, Prompts for reproducible workflows, and Context for logging."*